In [ ]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

<h4>LLM intergrates with tools run in a loop to acheive a goal</h4>

In [2]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [3]:
openai = OpenAI()

In [4]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [5]:
todos = []
completed = []

In [6]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [7]:
get_todo_report()

''

In [8]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [9]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [10]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [11]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [13]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [14]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [15]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [16]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [17]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-5.2", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [18]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [19]:
todos, completed = [], []
loop(messages)

Todo #1: Set up variables, distances, and timeline; assume standard Boston–New York distance since not provided.
Todo #2: Compute relative motion after both trains are moving; account for first train’s 1-hour head start.
Todo #3: Solve for meeting time and express in clock time.
Todo #4: Sanity-check result and present final answer.

Assumption needed: The BostonNew York distance isnt given, so use a reasonable standard estimate of 215 miles 
(approx. driving distance / common rail corridor scale). 
Timeline: Train B (Boston) leaves at 2:00 pm at 60 mph. Train N (NY) leaves at 3:00 pm at 80 mph toward Boston.

Todo #1: Set up variables, distances, and timeline; assume standard Boston–New York distance since not provided.
Todo #2: Compute relative motion after both trains are moving; account for first train’s 1-hour head start.
Todo #3: Solve for meeting time and express in clock time.
Todo #4: Sanity-check result and present final answer.

By 3:00 pm, the Boston train has a 1-hour head start: distance covered = 60 mph  1 h = 60 miles. Remaining 
separation at 3:00 pm: 215  60 = 155 miles.
From 3:00 pm onward, they approach each other with relative speed 60+80 = 140 mph.

Todo #1: Set up variables, distances, and timeline; assume standard Boston–New York distance since not provided.
Todo #2: Compute relative motion after both trains are moving; account for first train’s 1-hour head start.
Todo #3: Solve for meeting time and express in clock time.
Todo #4: Sanity-check result and present final answer.

Time after 3:00 pm to meet = remaining distance / relative speed = 155/140 h = 31/28 h  1.1071 h = 1 h 6 min 26 s 
(approx). 
Meeting time = 3:00 pm + 1:06:26  4:06 pm.

Todo #1: Set up variables, distances, and timeline; assume standard Boston–New York distance since not provided.
Todo #2: Compute relative motion after both trains are moving; account for first train’s 1-hour head start.
Todo #3: Solve for meeting time and express in clock time.
Todo #4: Sanity-check result and present final answer.

Check: Total time from 2:00 pm to meet is 1 h + 1.1071 h = 2.1071 h, so Boston train travels 60*2.1071  126.4 mi; 
NY train travels 80*1.1071  88.6 mi; sum  215 mi. Consistent.

Todo #1: Set up variables, distances, and timeline; assume standard Boston–New York distance since not provided.
Todo #2: Compute relative motion after both trains are moving; account for first train’s 1-hour head start.
Todo #3: Solve for meeting time and express in clock time.
Todo #4: Sanity-check result and present final answer.

Assuming Boston–New York is about 215 miles apart (distance not given):

• By 3:00 pm, the Boston train has gone 60 miles (60 mph × 1 hr), leaving 155 miles between them.  
• After 3:00 pm they approach each other at 140 mph (60 + 80).  
• Time to meet after 3:00 pm: 155/140 hr = 1.1071 hr ≈ 1 hr 6 min.

They meet at about 4:06 pm.